# Rental Property Research Agent

Week 1's `rag` function always did the same thing: search a cached index, build a prompt, call the LLM once. This notebook turns it into an agent that decides which tools to call, looks at the results, and keeps going until it can fill in the property record.

The structure follows the course's `documentation-agent`: a tools class whose methods become the agent's tools (`tools.py`), a PydanticAI `Agent` that runs the tool-calling loop, and a callback that prints each tool call (`doc_agent.py`).

In [ ]:
import json
import re
from dataclasses import dataclass
from typing import Any, Dict, List, Optional
from urllib.parse import urlparse

import requests
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
from pydantic_ai import Agent
from pydantic_ai.messages import FunctionToolCallEvent, FunctionToolResultEvent

openai_client = OpenAI()

## 1. Tools

| Tool | What it does |
| --- | --- |
| `lookup_property` | Standardizes a US address (Census geocoder) and returns the official owner, manager, and unit records. NYC only for now: HPD registrations and PLUTO, queried live by tax lot (BBL). |
| `find_portfolio` | Lists NYC buildings where a company or person appears as owner, manager, or officer. |
| `web_search` | Searches the web with OpenAI's built-in web search tool (no extra API key). |
| `fetch_page` | Reads a page through the Jina reader and returns its text and links. |
| `detect_pm_software` | Matches resident portal domains (AppFolio, RentCafe, Entrata, ...) on a URL or a page's links. |

`lookup_property` replaces week 1's `search`. It does the same job (address in, building record out), but instead of returning the top 5 TF-IDF matches from one cached zip code, it resolves the address to its exact tax lot and queries the city's data live, so it covers all of NYC and can't hand the model a neighboring building by mistake. It is also the place to add more counties later: each new source plugs in behind the same tool.

Tools return error messages instead of raising, so the agent can recover and try something else.

In [ ]:
SOCRATA = 'https://data.cityofnewyork.us/resource'
HPD_REGISTRATIONS = f'{SOCRATA}/tesw-yqqr.json'
HPD_CONTACTS = f'{SOCRATA}/feu5-w2e2.json'
HPD_BUILDINGS = f'{SOCRATA}/kj4p-ruqc.json'
PLUTO = f'{SOCRATA}/64uk-42ks.json'

CENSUS_GEOCODER = 'https://geocoding.geo.census.gov/geocoder/geographies/onelineaddress'
NYC_GEOSEARCH = 'https://geosearch.planninglabs.nyc/v2/search'
JINA_READER = 'https://r.jina.ai'

# County FIPS codes for the five boroughs -> HPD borough id
NYC_COUNTIES = {'36005': '2', '36047': '3', '36061': '1', '36081': '4', '36085': '5'}

CONTACT_ROLES = {
    'owner': ['CorporateOwner', 'IndividualOwner', 'JointOwner'],
    'manager': ['Agent', 'SiteManager'],
    'officer': ['HeadOfficer', 'Officer'],
}

# Resident portal domains -> property management software
PORTAL_DOMAINS = {
    'appfolio.com': 'AppFolio',
    'rentcafe.com': 'Yardi (RentCafe)',
    'securecafe.com': 'Yardi (RentCafe)',
    'yardi.com': 'Yardi',
    'entrata.com': 'Entrata',
    'residentportal.com': 'Entrata',
    'buildium.com': 'Buildium',
    'managebuilding.com': 'Buildium',
    'rentmanager.com': 'Rent Manager',
    'propertyware.com': 'Propertyware',
    'realpage.com': 'RealPage',
    'loftliving.com': 'RealPage',
    'myresman.com': 'ResMan',
    'tenantcloud.com': 'TenantCloud',
    'doorloop.com': 'DoorLoop',
    'rentvine.com': 'Rentvine',
}


def socrata_get(url, **params):
    params.setdefault('$limit', 1000)
    response = requests.get(url, params=params, timeout=60)
    response.raise_for_status()
    return response.json()


def sql_list(values):
    return ','.join("'" + str(v).replace("'", "''") + "'" for v in values)


def format_contact(c):
    person = ' '.join(p for p in [c.get('firstname'), c.get('lastname')] if p)
    company = c.get('corporationname', '')
    if company and person:
        name = f'{company} ({person})'
    else:
        name = company or person
    business_address = ' '.join(
        p for p in [c.get('businesshousenumber'), c.get('businessstreetname'),
                    c.get('businesscity'), c.get('businessstate')] if p
    )
    return {'type': c['type'], 'name': name, 'business_address': business_address}


def match_portal_domain(url):
    host = urlparse(url).netloc.lower()
    for domain, software in PORTAL_DOMAINS.items():
        if host == domain or host.endswith('.' + domain):
            return software
    return None

In [ ]:
class PropertyTools:
    """
    Tools for researching who owns and manages a US rental property.
    """

    def __init__(self, openai_client: OpenAI, web_search_model: str = 'gpt-4o-mini') -> None:
        self.openai_client = openai_client
        self.web_search_model = web_search_model

    def lookup_property(self, address: str) -> Dict[str, Any]:
        """
        Look up the official public records for one property address.

        Standardizes the address with the US Census geocoder. For NYC addresses,
        returns the HPD rental registration (owner, officers, managing agent,
        site manager, legal unit count) and the PLUTO tax lot record (owner of
        record, building class, year built). Other counties aren't covered yet.

        Args:
            address: Full street address, e.g. "123 Linden Blvd, Brooklyn, NY 11226".

        Returns:
            The records found, with source URLs to cite as evidence, or a
            "found": false result explaining why nothing was returned.
        """
        geocode = requests.get(CENSUS_GEOCODER, params={
            'address': address,
            'benchmark': 'Public_AR_Current',
            'vintage': 'Current_Current',
            'layers': 'Counties',
            'format': 'json',
        }, timeout=60).json()

        matches = geocode['result']['addressMatches']
        if not matches:
            return {'found': False, 'reason': f'The Census geocoder could not match "{address}".'}

        match = matches[0]
        county = match['geographies']['Counties'][0]
        standardized_address = match['matchedAddress']

        if county['GEOID'] not in NYC_COUNTIES:
            return {
                'found': False,
                'standardized_address': standardized_address,
                'county': f"{county['NAME']} (FIPS {county['GEOID']})",
                'reason': 'No public records source for this county yet. Use web_search instead.',
            }

        geosearch = requests.get(NYC_GEOSEARCH, params={'text': standardized_address, 'size': 1}, timeout=60).json()
        if not geosearch['features']:
            return {'found': False, 'standardized_address': standardized_address,
                    'reason': 'NYC Geosearch could not find a tax lot for this address.'}

        bbl = geosearch['features'][0]['properties']['addendum']['pad']['bbl']
        boro_id, block, lot = bbl[0], str(int(bbl[1:6])), str(int(bbl[6:]))

        lots = socrata_get(PLUTO, bbl=bbl)
        tax_lot = lots[0] if lots else {}

        registrations = socrata_get(HPD_REGISTRATIONS, boroid=boro_id, block=block, lot=lot)
        registration_ids = [r['registrationid'] for r in registrations]

        contacts, buildings = [], []
        if registration_ids:
            contacts = socrata_get(HPD_CONTACTS, **{'$where': f'registrationid in ({sql_list(registration_ids)})'})
            buildings = socrata_get(HPD_BUILDINGS, **{'$where': f'registrationid in ({sql_list(registration_ids)})'})
        buildings_by_id = {b['buildingid']: b for b in buildings}

        hpd_registrations = []
        for r in registrations:
            building = buildings_by_id.get(r['buildingid'], {})
            hpd_registrations.append({
                'registration_id': r['registrationid'],
                'address': f"{r.get('housenumber', '')} {r.get('streetname', '')}, {r['boro']}, NY {r.get('zip', '')}",
                'last_registration_date': r.get('lastregistrationdate', '')[:10],
                'legal_units': int(building.get('legalclassa') or 0) + int(building.get('legalclassb') or 0),
                'contacts': [format_contact(c) for c in contacts if c['registrationid'] == r['registrationid']],
                'source_url': f"{HPD_CONTACTS}?registrationid={r['registrationid']}",
            })

        return {
            'found': True,
            'standardized_address': standardized_address,
            'county': county['NAME'],
            'bbl': bbl,
            'tax_lot': {
                'owner_of_record': tax_lot.get('ownername'),
                'building_class': tax_lot.get('bldgclass'),
                'residential_units': tax_lot.get('unitsres'),
                'year_built': tax_lot.get('yearbuilt'),
                'source_url': f'{PLUTO}?bbl={bbl}',
            },
            'hpd_registrations': hpd_registrations or
                'No HPD registration. Buildings with fewer than 3 units usually are not required to register.',
        }

    def find_portfolio(self, entity_name: str, role: Optional[str] = None) -> Dict[str, Any]:
        """
        List NYC rental buildings where a company or person appears in the HPD
        registration as owner, manager, or officer.

        Use it to answer portfolio questions ("what does X manage?") and to
        link single-purpose LLCs to a parent through a shared officer.

        Args:
            entity_name: Company or person name, or a distinctive part of it,
                e.g. "Meridian Properties" or "Aidin Ujka".
            role: Optional filter: "owner", "manager", or "officer".

        Returns:
            Matching buildings with the role and exact name on each registration.
        """
        name = entity_name.upper().replace("'", "''")
        where = (
            f"(upper(corporationname) like '%{name}%' "
            f"or upper(firstname || ' ' || lastname) like '%{name}%')"
        )
        if role:
            if role not in CONTACT_ROLES:
                return {'error': f'role must be one of {list(CONTACT_ROLES)}'}
            where += f' and type in ({sql_list(CONTACT_ROLES[role])})'

        contacts = socrata_get(HPD_CONTACTS, **{'$where': where, '$limit': 500})
        if not contacts:
            return {'entity_name': entity_name, 'total_buildings': 0, 'buildings': []}

        roles_by_registration = {}
        for c in contacts:
            contact = format_contact(c)
            roles_by_registration.setdefault(c['registrationid'], set()).add(f"{contact['type']}: {contact['name']}")

        registration_ids = list(roles_by_registration)
        registrations = []
        for i in range(0, len(registration_ids), 200):
            batch = registration_ids[i:i + 200]
            registrations.extend(socrata_get(HPD_REGISTRATIONS, **{'$where': f'registrationid in ({sql_list(batch)})'}))

        buildings = [{
            'address': f"{r.get('housenumber', '')} {r.get('streetname', '')}, {r['boro']}, NY {r.get('zip', '')}",
            'roles': sorted(roles_by_registration[r['registrationid']]),
            'registration_id': r['registrationid'],
        } for r in registrations]

        return {
            'entity_name': entity_name,
            'total_buildings': len(buildings),
            'buildings': buildings[:50],
            'note': 'Showing the first 50.' if len(buildings) > 50 else '',
        }

    def web_search(self, query: str) -> Dict[str, Any]:
        """
        Search the web.

        Use it to find a property's website or resident portal, a management
        company's website, or the company behind an LLC.

        Args:
            query: The search query. Include the city and state to avoid
                companies with the same name elsewhere.

        Returns:
            A short summary of what the search found and the source URLs.
        """
        response = self.openai_client.responses.create(
            model=self.web_search_model,
            tools=[{'type': 'web_search'}],
            input=query,
            include=['web_search_call.action.sources'],
        )

        sources = []
        for item in response.output:
            if item.type == 'web_search_call' and item.action.sources:
                sources.extend(s.url.replace('?utm_source=openai', '') for s in item.action.sources)

        return {'summary': response.output_text, 'sources': list(dict.fromkeys(sources))}

    def fetch_page(self, url: str) -> Dict[str, Any]:
        """
        Read a web page.

        Args:
            url: The page URL.

        Returns:
            The page text (truncated to 8000 characters) and the links on
            the page. Resident portal links are the evidence for which
            property management software a company uses.
        """
        try:
            response = requests.get(
                f'{JINA_READER}/{url}',
                headers={'X-With-Links-Summary': 'true'},
                timeout=60,
            )
            response.raise_for_status()
        except requests.RequestException as e:
            return {'url': url, 'error': str(e)}

        content = response.text
        text, _, links_section = content.partition('Links/Buttons:')
        links = re.findall(r'\[([^\]]*)\]\((https?://[^)\s]+)\)', links_section)

        return {
            'url': url,
            'text': text[:8000],
            'links': [{'text': t, 'url': u} for t, u in dict.fromkeys(links)][:150],
        }

    def detect_pm_software(self, url: str) -> Dict[str, Any]:
        """
        Identify the property management software behind a property or
        management company website by matching known resident portal domains
        (AppFolio, Yardi/RentCafe, Entrata, Buildium, Rent Manager, and others).

        Pass a resident portal URL directly, or a website or "Pay Rent" page;
        the page's links are checked for portal domains.

        Args:
            url: A portal URL or a page that links to the portal.

        Returns:
            The software name, the URLs that prove it, and a confidence level.
        """
        software = match_portal_domain(url)
        if software:
            return {'software': software, 'evidence_urls': [url], 'confidence': 'high'}

        page = self.fetch_page(url)
        if 'error' in page:
            return {'software': None, 'error': page['error']}

        found = {}
        for link in page['links']:
            software = match_portal_domain(link['url'])
            if software:
                found.setdefault(software, []).append(link['url'])

        if not found:
            return {
                'software': None,
                'checked_url': url,
                'note': 'No known portal domain linked from this page. Try its "Pay Rent" or "Residents" page.',
            }

        best = max(found, key=lambda s: len(found[s]))
        return {
            'software': best,
            'evidence_urls': found[best][:3],
            'confidence': 'high' if len(found) == 1 else 'medium',
            'other_matches': {s: urls[:3] for s, urls in found.items() if s != best},
        }

In [ ]:
property_tools = PropertyTools(openai_client)

property_tools.lookup_property('481 E 15th St, Brooklyn, NY')

## 2. Instructions

In [ ]:
INSTRUCTIONS = """
You're a research agent for a team that builds models of the US residential
rental market. They copy your answers into their system of record, so every
fact needs evidence they can check.

Users ask four kinds of questions:
1. Address lookup: who owns and manages a property, and what kind of property it is
2. Portfolio: which buildings a company or person owns or manages
3. Software: which property management software a property or manager uses
4. Owner normalization: which company or person is behind an LLC

When to use each tool:
- lookup_property: always call it first when the question contains an address.
  It covers NYC only for now. Elsewhere it returns found: false, and you
  research the property with web_search instead.
- find_portfolio: for portfolio questions, and to find the parent behind an
  owner LLC. Search the LLC's head officer or managing agent and look at
  which other LLCs share them.
- web_search: to find the managing agent's or property's website and
  resident portal, or who is behind an LLC. Always include the city and state.
- fetch_page: to read a page that web_search found, e.g. to confirm a
  company's address or to find its "Pay Rent" or "Residents" page.
- detect_pm_software: on a resident portal URL or a "Pay Rent"/"Residents"
  page. Never name the software based only on a search summary.

Workflow for an address lookup:
1. lookup_property.
2. Take the owner, officers, managing agent, and units from the records.
3. Normalized owner: if the owner is a single-purpose LLC, call
   find_portfolio on its head officer, then web_search the officer or the
   business address together with the city to name the parent company.
4. Software: web_search for the managing agent's resident portal or
   "pay rent" page, then call detect_pm_software on the best URL.
5. Budget: at most 10 tool calls in total, and at most 4 of them on the
   software search. When the budget runs out, answer with what you have
   and mark the rest unknown.

Checking web results:
Many companies share a name (there is a Columbus Property Management in New
York and another in Ohio). Only use a web result when it matches the
property's city or the business address in the records. If it doesn't,
say so and don't use it.

Answer format for an address lookup:

Property: <standardized address>
- Property type: <value> | evidence: <source> | confidence: <level>
- Owner of record: ...
- Normalized owner: ...
- Owner type: ...
- Property manager: ...
- Property management software: ...
- Number of units: ...
Conflicts and gaps: <records that disagree, fields left unknown and what
would resolve them>

Field rules:
- Property type: from the building class (A = one-family, B = two-family,
  C = walk-up apartments, D = elevator apartments, R = condo, S = mixed
  residential and retail) and the unit count.
- Owner type: institutional (large national or private-equity-backed
  portfolio), mid-size (regional operator with many buildings), or
  individual (a person or family). Being an LLC says nothing about owner
  type. Base it on the size of the portfolio you found.
- Property manager: the managing agent, or the site manager if there's
  no agent.

Evidence is a record ID or URL from a tool result: an HPD registration ID,
a BBL, or a web page URL. Confidence:
- high: an official record or a portal link states it directly
- medium: inferred from links between records, such as a shared officer
- low: a single web source or a weak match
If no tool result supports a value, write "unknown". Never fill a value
from your own knowledge.

For portfolio questions, list the buildings with their roles and the total
count, and say the list only covers NYC HPD registrations.
""".strip()

## 3. Agent

PydanticAI runs the agent loop: it sends the question and tool definitions to the model, executes the tool calls the model asks for, sends the results back, and repeats until the model gives a final answer. The tool descriptions it sends come from the docstrings above.

In [ ]:
@dataclass
class PropertyAgentConfig:
    model: str = 'openai:gpt-5-mini'
    name: str = 'property_research'
    instructions: str = INSTRUCTIONS


def create_agent(config: PropertyAgentConfig, property_tools: PropertyTools) -> Agent:
    tools = [
        property_tools.lookup_property,
        property_tools.find_portfolio,
        property_tools.web_search,
        property_tools.fetch_page,
        property_tools.detect_pm_software,
    ]

    return Agent(
        name=config.name,
        model=config.model,
        instructions=config.instructions,
        tools=tools,
    )


agent = create_agent(PropertyAgentConfig(), property_tools)

In [ ]:
class NamedCallback:

    def __init__(self, agent):
        self.agent_name = agent.name

    async def print_function_calls(self, ctx, event):
        # Detect nested streams
        if hasattr(event, "__aiter__"):
            async for sub in event:
                await self.print_function_calls(ctx, sub)
            return

        if isinstance(event, FunctionToolCallEvent):
            tool_name = event.part.tool_name
            args = event.part.args
            print(f"TOOL CALL ({self.agent_name}): {tool_name}({args})")

        if isinstance(event, FunctionToolResultEvent):
            tool_name = event.part.tool_name
            size = len(json.dumps(event.part.content, default=str))
            print(f"TOOL RESULT ({self.agent_name}): {tool_name} returned {size} characters")

    async def __call__(self, ctx, event):
        return await self.print_function_calls(ctx, event)


async def run_agent(agent: Agent, user_prompt: str, message_history=None):
    callback = NamedCallback(agent)

    if message_history is None:
        message_history = []

    print(f"USER PROMPT ({agent.name}): {user_prompt}")

    result = await agent.run(
        user_prompt,
        event_stream_handler=callback,
        message_history=message_history,
    )

    print(f"DONE ({agent.name}): {result.usage}")

    return result

## 4. Try It Out

An address lookup, the most common request:

In [ ]:
result = await run_agent(
    agent,
    "Who owns and manages 123 Linden Blvd, Brooklyn, NY 11226, and what property management software do they use?",
)

In [ ]:
print(result.output)

A portfolio question:

In [ ]:
result = await run_agent(agent, "Which NYC buildings is Aidin Ujka connected to, and in what role?")

In [ ]:
print(result.output)